# 02 - Visualization: How AI Changed Human Chess

Load preprocessed data from notebook 01 and visualize the impact of AI breakthroughs
(AlphaZero 2017, Stockfish NNUE 2020) on human chess playing patterns.

Visualizations:
1. Opening tree (Sunburst partition chart, toggle between periods)
2. Opening revolution (top named openings across periods)
3. Material curve + sacrifice rate
4. Blunder rate by ELO bracket over time
5. Piece-square heatmap (pre-AI vs modern)
6. Game length distribution

## Setup
Install dependencies and load data.

In [ ]:
import subprocess, sys

def install(pkg):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", pkg])

install("plotly")
install("seaborn")

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from collections import Counter
import os

plt.rcParams['figure.dpi'] = 100
plt.rcParams['figure.figsize'] = (12, 6)
sns.set_style("whitegrid")

PERIOD_ORDER = ["pre-ai", "early-post-ai", "nnue-era", "modern"]
PERIOD_LABELS = {
    "pre-ai": "Pre-AI (2015-16)",
    "early-post-ai": "Early Post-AI (2018-19)",
    "nnue-era": "NNUE Era (2021-22)",
    "modern": "Modern (2023-25)",
}
colors_period = dict(zip(PERIOD_ORDER, sns.color_palette("viridis", 4)))

def periods_in_data(df):
    """Return only periods from PERIOD_ORDER that exist in the dataframe."""
    available = set(df['period'].unique()) if 'period' in df.columns else set()
    return [p for p in PERIOD_ORDER if p in available]

print("Setup done.")

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

DATA_DIR = "/content/drive/MyDrive/Learning-document/Data Visualization/Project 2/data"

games = pd.read_csv(os.path.join(DATA_DIR, "lichess_sampled_games.csv"))
moves = pd.read_csv(os.path.join(DATA_DIR, "lichess_sampled_moves.csv"))
blunders = pd.read_csv(os.path.join(DATA_DIR, "lichess_sampled_blunders.csv"))
piece_squares = pd.read_csv(os.path.join(DATA_DIR, "lichess_sampled_piece_squares.csv"))
material_curve = pd.read_csv(os.path.join(DATA_DIR, "lichess_sampled_material_curve.csv"))

print(f"Games: {len(games):,}")
print(f"Moves (eval): {len(moves):,}")
print(f"Blunders: {len(blunders):,}")
print(f"Piece squares: {len(piece_squares):,}")
print(f"Material curve: {len(material_curve):,}")
print(f"\nPeriods: {games['period'].value_counts().to_dict()}")

## Frontend Data Export

Aggregate the 5 raw CSVs into small files for the React frontend.
Output goes to `data/frontend/` on Drive. Copy to `frontend/public/data/` for deployment.

In [ ]:
import json

PERIOD_DISPLAY = {
    'pre-ai': 'Pre-AI',
    'early-post-ai': 'Early Post-AI',
    'nnue-era': 'NNUE Era',
    'modern': 'Modern',
}
ELO_ORDER = ['0-1000', '1000-1400', '1400-1800', '1800-2200', '2200-2600', '2600+']
FRONTEND_DIR = os.path.join(DATA_DIR, "frontend")
os.makedirs(FRONTEND_DIR, exist_ok=True)

# === 1. Opening Tree JSON (one per period, pruned) ===
print("1. Opening tree JSON (pruned)...")

def build_tree(df_period, max_ply=10):
    root = {'san': 'root', 'children': []}
    for moves_str in df_period['first_10_moves'].dropna():
        moves = moves_str.split(',')
        node = root
        for move in moves[:max_ply]:
            child = next((c for c in node.get('children', []) if c['san'] == move), None)
            if not child:
                child = {'san': move, 'count': 0, 'children': []}
                node.setdefault('children', []).append(child)
            child['count'] += 1
            node = child
    def clean(n):
        if 'children' in n and len(n['children']) == 0:
            del n['children']
        elif 'children' in n:
            for c in n['children']:
                clean(c)
    clean(root)
    return root

def subtree_value(node):
    if 'children' not in node or not node['children']:
        return node.get('count', 0)
    return sum(subtree_value(c) for c in node['children'])

def prune_tree(node, depth=0):
    children = node.get('children', [])
    if not children:
        return node
    if depth <= 2:
        limit, create_other = 5, False
    elif depth <= 4:
        limit, create_other = 3, True
    else:
        limit, create_other = 2, False
    sorted_children = sorted(children, key=lambda c: c.get('count', 0), reverse=True)
    kept = [prune_tree(c, depth + 1) for c in sorted_children[:limit]]
    if create_other and len(sorted_children) > limit:
        excluded = sorted_children[limit:]
        other_count = sum(subtree_value(c) for c in excluded)
        if other_count > 0:
            kept.append({'san': 'Other', 'count': other_count})
    result = {'san': node['san'], 'count': node.get('count', 0)}
    if kept:
        result['children'] = kept
    return result

def count_nodes(n):
    return 1 + sum(count_nodes(c) for c in n.get('children', []))

for period in PERIOD_ORDER:
    df_p = games[games['period'] == period]
    tree = build_tree(df_p)
    raw_nodes = count_nodes(tree)
    tree = prune_tree(tree)
    pruned_nodes = count_nodes(tree)
    path = os.path.join(FRONTEND_DIR, f'opening_tree_{period}.json')
    with open(path, 'w') as f:
        json.dump(tree, f)
    size_kb = os.path.getsize(path) / 1024
    n_top = len(tree.get('children', []))
    print(f"  {period}: {len(df_p):,} games, {raw_nodes:,} -> {pruned_nodes:,} nodes ({size_kb:.1f} KB)")

# === 2. First Move by Period ===
print("\n2. First move by period...")
rows = []
for period in PERIOD_ORDER:
    df_p = games[games['period'] == period]
    total = len(df_p)
    fm_counts = df_p['first_move'].value_counts()
    row = {'period': PERIOD_DISPLAY[period]}
    for move in ['e4', 'd4', 'c4', 'Nf3']:
        row[move] = round(fm_counts.get(move, 0) / total * 100, 1)
    row['other'] = round(100 - sum(row[m] for m in ['e4', 'd4', 'c4', 'Nf3']), 1)
    rows.append(row)
pd.DataFrame(rows).to_csv(os.path.join(FRONTEND_DIR, 'first_move_by_period.csv'), index=False)
print("  Done")

# === 3. Material Curve (wide format) ===
print("\n3. Material curve...")
mc = material_curve.copy()
mc['total_material'] = mc['avg_white_material'] + mc['avg_black_material']
mat_wide = mc.pivot(index='ply', columns='period', values='total_material')
mat_wide.columns = [PERIOD_DISPLAY.get(c, c) for c in mat_wide.columns]
mat_wide = mat_wide.reset_index()
mat_wide.to_csv(os.path.join(FRONTEND_DIR, 'material_total.csv'), index=False)
print(f"  {len(mat_wide)} rows")

# === 4. Sacrifice Rate ===
print("\n4. Sacrifice rate...")
sac_rows = []
for period in PERIOD_ORDER:
    df_p = games[games['period'] == period]
    sac_rows.append({
        'period': PERIOD_DISPLAY[period],
        'avgSacrifices': round(df_p['sacrifice_count'].mean(), 2),
    })
pd.DataFrame(sac_rows).to_csv(os.path.join(FRONTEND_DIR, 'sacrifice_rate.csv'), index=False)
print("  Done")

# === 5. Blunder Rate ===
print("\n5. Blunder rate...")
bpg = blunders.groupby('game_idx').size().rename('blunder_count')
games_bl = games.join(bpg)
games_bl['blunder_count'] = games_bl['blunder_count'].fillna(0).astype(int)

blunder_rows = []
for period in PERIOD_ORDER:
    for bracket in ELO_ORDER:
        mask = (games_bl['period'] == period) & (games_bl['elo_bracket'] == bracket)
        subset = games_bl[mask]
        if len(subset) > 0:
            blunder_rows.append({
                'elo': bracket,
                'period': PERIOD_DISPLAY[period],
                'value': round(subset['blunder_count'].mean(), 2),
            })
pd.DataFrame(blunder_rows).to_csv(os.path.join(FRONTEND_DIR, 'blunder_rate.csv'), index=False)
print(f"  {len(blunder_rows)} cells")

# === 6. Piece Squares ===
print("\n6. Piece squares...")
PIECE_LETTER = {'knight': 'N', 'bishop': 'B', 'rook': 'R', 'queen': 'Q', 'king': 'K', 'pawn': 'P'}
psq = piece_squares.copy()
psq['piece'] = psq['piece'].map(PIECE_LETTER)
psq_agg = psq.groupby(['piece', 'period', 'square'])['count'].sum().reset_index()
psq_agg.to_csv(os.path.join(FRONTEND_DIR, 'piece_squares_agg.csv'), index=False)
print(f"  {len(psq_agg)} rows")

# === 7. Game Length (wide format) ===
print("\n7. Game length...")
gl_rows = []
for period in PERIOD_ORDER:
    df_p = games[games['period'] == period]
    counts = df_p['game_length'].value_counts()
    for ply, count in counts.items():
        gl_rows.append({'ply': int(ply), 'period': PERIOD_DISPLAY[period], 'count': int(count)})
gl_df = pd.DataFrame(gl_rows)
gl_wide = gl_df.pivot(index='ply', columns='period', values='count').fillna(0).astype(int).reset_index()
gl_wide.to_csv(os.path.join(FRONTEND_DIR, 'game_length.csv'), index=False)
print(f"  {len(gl_wide)} ply values")

# === 8. Opening by Year ===
print("\n8. Opening by year...")
opening_year = games.groupby(['opening_name', 'year', 'period']).size().reset_index(name='count')
opening_year.to_csv(os.path.join(FRONTEND_DIR, 'opening_by_year.csv'), index=False)
print(f"  {len(opening_year):,} rows")

print("\nAll exports done.")

In [ ]:
# === 9. Opening by Period (for Opening Revolution chart) ===
print("\n9. Opening by period...")

# Extract base opening name (before colon/pipe)
games['opening_base'] = games['opening_name'].str.split(':').str[0].str.split('|').str[0].str.strip()

# Get top 10 openings overall (excluding modern if sparse)
avail_periods_opening = [p for p in PERIOD_ORDER if len(games[games['period'] == p]) > 100]
all_opening_counts = games[games['period'].isin(avail_periods_opening)].groupby('opening_base').size()
top_openings = all_opening_counts.nlargest(10).index.tolist()

opening_rows = []
for period in PERIOD_ORDER:
    df_p = games[games['period'] == period]
    total = len(df_p)
    for opening in top_openings:
        count = len(df_p[df_p['opening_base'] == opening])
        pct = round(count / total * 100, 2) if total > 0 else 0
        opening_rows.append({
            'period': period,
            'opening': opening,
            'count': count,
            'pct': pct,
        })

pd.DataFrame(opening_rows).to_csv(os.path.join(FRONTEND_DIR, 'opening_by_period.csv'), index=False)
print(f"  {len(opening_rows)} rows, top openings: {top_openings[:5]}...")


In [ ]:
# Export summary
print("Files exported to:", FRONTEND_DIR)
print()
for f in sorted(os.listdir(FRONTEND_DIR)):
    path = os.path.join(FRONTEND_DIR, f)
    size = os.path.getsize(path)
    if size > 1024 * 1024:
        print(f"  {f} ({size / 1024**2:.1f} MB)")
    elif size > 1024:
        print(f"  {f} ({size / 1024:.1f} KB)")
    else:
        print(f"  {f} ({size} B)")

total_size = sum(os.path.getsize(os.path.join(FRONTEND_DIR, f)) for f in os.listdir(FRONTEND_DIR))
print(f"\n  Total: {total_size / 1024:.1f} KB")
print(f"\nCopy these to frontend/public/data/ for deployment.")

## Visualizations

The cells below are for in-notebook exploration. The frontend data export above is independent.

In [ ]:
# First move by period
avail = periods_in_data(games)
fm_period = pd.crosstab(games['first_move'], games['period'], normalize='columns') * 100
fm_period = fm_period[avail]
fm_top = fm_period.head(5)

fig, ax = plt.subplots(figsize=(10, 6))
x = np.arange(len(avail))
width = 0.15
colors_fm = {'e4': '#e74c3c', 'd4': '#3498db', 'c4': '#2ecc71', 'Nf3': '#f39c12', 'g3': '#9b59b6'}

for i, move in enumerate(fm_top.index):
    offset = (i - len(fm_top)/2 + 0.5) * width
    ax.bar(x + offset, fm_top.loc[move], width, label=move,
           color=colors_fm.get(move, None), alpha=0.85)

ax.set_ylabel('% of Games')
ax.set_title('First Move Distribution Across Periods')
ax.set_xticks(x)
ax.set_xticklabels([PERIOD_LABELS[p] for p in avail])
ax.legend(title='First Move')
plt.tight_layout()
plt.show()

print("\nFirst move % by period:")
print(fm_top.round(1).to_string())

## 3. Material Curve + Sacrifices

Average material on the board at each ply, compared across periods.
Also: sacrifice rate over time - did humans become bolder after watching engines sacrifice?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

avail_mat = periods_in_data(material_curve)

# Total material on board
for period in avail_mat:
    mc = material_curve[material_curve['period'] == period].sort_values('ply')
    total_mat = mc['avg_white_material'] + mc['avg_black_material']
    ax1.plot(mc['ply'], total_mat, label=PERIOD_LABELS[period],
             color=colors_period[period], linewidth=2)

ax1.set_xlabel('Ply (half-moves)')
ax1.set_ylabel('Total Material on Board')
ax1.set_title('Material Decay Over Time')
ax1.set_xlim(0, 150)
ax1.legend(fontsize=9)

# Material difference (White - Black)
for period in avail_mat:
    mc = material_curve[material_curve['period'] == period].sort_values('ply')
    diff = mc['avg_white_material'] - mc['avg_black_material']
    ax2.plot(mc['ply'], diff, label=PERIOD_LABELS[period],
             color=colors_period[period], linewidth=2)

ax2.set_xlabel('Ply (half-moves)')
ax2.set_ylabel('Material Difference (White - Black)')
ax2.set_title('Material Balance Over Time')
ax2.axhline(y=0, color='black', linestyle='--', alpha=0.3)
ax2.set_xlim(0, 150)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

In [ ]:
# Sacrifice rate by period
avail = periods_in_data(games)

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Avg sacrifices per game
sac_by_period = games.groupby('period')['sacrifice_count'].mean().reindex(avail)
axes[0].bar(range(len(avail)), sac_by_period.values, color=[colors_period[p] for p in avail])
axes[0].set_xticks(range(len(avail)))
axes[0].set_xticklabels([PERIOD_LABELS[p] for p in avail], fontsize=9)
axes[0].set_ylabel('Avg Sacrifices / Game')
axes[0].set_title('Sacrifice Rate Across Periods')

# Sacrifice rate by ELO bracket
sac_elo = games.pivot_table(values='sacrifice_count', index='elo_bracket', columns='period', aggfunc='mean')
sac_elo = sac_elo.reindex(columns=avail)
sac_elo.plot(kind='bar', ax=axes[1], color=[colors_period[p] for p in avail], width=0.8)
axes[1].set_ylabel('Avg Sacrifices / Game')
axes[1].set_xlabel('ELO Bracket')
axes[1].set_title('Sacrifice Rate by ELO and Period')
axes[1].legend([PERIOD_LABELS[p] for p in avail], fontsize=8)
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout()
plt.show()

## 4. Blunder Rate by ELO Over Time

Did AI coaching tools (post-2017) help lower-rated players reduce blunders?
Heatmap of average blunders per game, by ELO bracket and period.

In [ ]:
# Count blunders per game
avail = periods_in_data(games)
bpg = blunders.groupby('game_idx').size()
games['blunder_count'] = games.index.map(lambda i: bpg.get(i, 0))

# Heatmap: ELO bracket x period
pivot = games.pivot_table(values='blunder_count', index='elo_bracket', columns='period', aggfunc='mean')
pivot = pivot.reindex(columns=avail)

# Sort ELO brackets properly
elo_order = [f"{b[0]}-{b[1]}" if b[1] < 9999 else f"{b[0]}+" for b in [
    (0,1000),(1000,1400),(1400,1800),(1800,2200),(2200,2600),(2600,9999)]]
pivot = pivot.reindex([e for e in elo_order if e in pivot.index])

fig, ax = plt.subplots(figsize=(10, 6))
sns.heatmap(pivot, annot=True, fmt='.2f', cmap='YlOrRd', ax=ax, linewidths=0.5)
ax.set_title('Average Blunders per Game by ELO and Period')
ax.set_ylabel('ELO Bracket')
ax.set_xlabel('')
ax.set_xticklabels([PERIOD_LABELS[p] for p in avail], rotation=0, fontsize=9)
ax.set_yticklabels(ax.get_yticklabels(), rotation=0)
plt.tight_layout()
plt.show()

# Also show blunder rate change (pre-ai -> latest available)
if 'pre-ai' in pivot.columns and len(avail) > 1:
    latest = avail[-1]
    print(f"\nBlunder rate change (pre-ai -> {latest}):")
    for bracket in pivot.index:
        pre = pivot.loc[bracket, 'pre-ai']
        mod = pivot.loc[bracket, latest]
        change = ((mod - pre) / pre * 100) if pre > 0 else 0
        print(f"  {bracket:>12}: {pre:.2f} -> {mod:.2f} ({change:+.1f}%)")

## 5. Piece-Square Heatmap

Where do pieces go? Comparing Pre-AI vs Modern piece placement patterns.
Darker squares = more frequent destinations.

In [ ]:
def plot_chess_heatmap(piece_squares_df, piece_name, is_white, periods):
    """Plot chess board heatmaps for a piece across periods."""
    avail_periods = [p for p in periods if p in set(piece_squares_df['period'].unique())]
    if not avail_periods:
        print(f"No data for {piece_name}, skipping")
        return

    fig, axes = plt.subplots(1, len(avail_periods), figsize=(5*len(avail_periods), 5))

    for ax, period in zip(axes if len(avail_periods) > 1 else [axes], avail_periods):
        mask = (piece_squares_df['piece'] == piece_name) & \
               (piece_squares_df['is_white'] == is_white) & \
               (piece_squares_df['period'] == period)
        data = piece_squares_df[mask]

        board = np.zeros((8, 8))
        for _, row in data.iterrows():
            sq = row['square']
            if len(sq) >= 2:
                file_idx = ord(sq[0]) - ord('a')
                rank_idx = int(sq[1]) - 1
                if 0 <= file_idx < 8 and 0 <= rank_idx < 8:
                    board[7 - rank_idx][file_idx] = row['count']

        # Chess board pattern
        checker = np.zeros((8, 8, 3))
        for r in range(8):
            for c in range(8):
                checker[r][c] = [0.93, 0.93, 0.93] if (r+c) % 2 == 0 else [0.5, 0.5, 0.5]

        ax.imshow(checker, extent=[-0.5, 7.5, -0.5, 7.5], aspect='equal')
        sns.heatmap(board, ax=ax, cmap='YlOrRd', annot=False, alpha=0.8,
                    xticklabels=list('abcdefgh'), yticklabels=list('87654321'),
                    cbar=True)
        ax.set_title(PERIOD_LABELS.get(period, period), fontsize=10)

    piece_label = f"White {piece_name.title()}" if is_white else f"Black {piece_name.title()}"
    fig.suptitle(f"{piece_label} - Square Utilization", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()


# Show key pieces: Knight, Bishop, Queen for White
avail = periods_in_data(piece_squares)
for piece in ['knight', 'bishop', 'queen']:
    plot_chess_heatmap(piece_squares, piece, True, avail)

In [ ]:
# Difference heatmap: Modern - Pre-AI (shows where piece placement shifted)
def plot_piece_diff(piece_squares_df, piece_name, is_white):
    """Show the difference in piece placement between modern and pre-ai."""
    fig, ax = plt.subplots(figsize=(6, 5))

    boards = {}
    for period in ["pre-ai", "modern"]:
        mask = (piece_squares_df['piece'] == piece_name) & \
               (piece_squares_df['is_white'] == is_white) & \
               (piece_squares_df['period'] == period)
        data = piece_squares_df[mask]
        if len(data) == 0:
            print(f"  No data for {piece_name} ({'White' if is_white else 'Black'}, {period}), skipping diff")
            return
        board = np.zeros((8, 8))
        total = data['count'].sum()
        for _, row in data.iterrows():
            sq = row['square']
            if len(sq) >= 2:
                fi = ord(sq[0]) - ord('a')
                ri = int(sq[1]) - 1
                if 0 <= fi < 8 and 0 <= ri < 8:
                    board[7 - ri][fi] = row['count'] / total * 100 if total > 0 else 0
        boards[period] = board

    if "pre-ai" not in boards or "modern" not in boards:
        print(f"  Cannot compute diff for {piece_name}: missing period data")
        return

    diff = boards["modern"] - boards["pre-ai"]
    max_abs = max(abs(diff.min()), abs(diff.max()), 0.5)
    sns.heatmap(diff, ax=ax, cmap='RdBu_r', center=0, vmin=-max_abs, vmax=max_abs,
                annot=True, fmt='.2f', annot_kws={'fontsize': 7},
                xticklabels=list('abcdefgh'), yticklabels=list('87654321'))
    piece_label = f"White {piece_name.title()}" if is_white else f"Black {piece_name.title()}"
    ax.set_title(f"{piece_label}: Modern - Pre-AI (pp)")
    plt.tight_layout()
    plt.show()


for piece in ['knight', 'bishop', 'queen']:
    plot_piece_diff(piece_squares, piece, True)

## 6. Game Length Distribution

How game length patterns changed. Watch for the 'comb' pattern at time control boundaries.
Did AI-era players adapt their time management?

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(16, 6))

avail = periods_in_data(games)

# Overlaid histograms
for period in avail:
    subset = games[games['period'] == period]
    subset['game_length'].hist(bins=80, alpha=0.35, label=PERIOD_LABELS[period],
                               ax=ax1, density=True, color=colors_period[period])
ax1.set_xlabel('Game Length (plies)')
ax1.set_ylabel('Density')
ax1.set_title('Game Length Distribution')
ax1.legend(fontsize=9)
ax1.set_xlim(0, 200)

# Comb pattern: line plot
for period in avail:
    subset = games[games['period'] == period]
    length_counts = subset['game_length'].value_counts().sort_index()
    if len(subset) > 0:
        ax2.plot(length_counts.index, length_counts.values / len(subset) * 100,
                 label=PERIOD_LABELS[period], color=colors_period[period],
                 linewidth=1.2, alpha=0.8)

# Annotate time control boundaries
for ply, label in [(80, 'Move 40\n(time ctrl)'), (120, 'Move 60')]:
    ax2.axvline(x=ply, color='gray', linestyle=':', alpha=0.4)
    ax2.annotate(label, xy=(ply, ax2.get_ylim()[1]*0.9), fontsize=8, color='gray', ha='center')

ax2.set_xlabel('Game Length (plies)')
ax2.set_ylabel('% of Games')
ax2.set_title('Game Length (Comb Pattern)')
ax2.set_xlim(20, 200)
ax2.legend(fontsize=9)

plt.tight_layout()
plt.show()

# Summary stats
print("\nGame length by period:")
for period in avail:
    subset = games[games['period'] == period]['game_length']
    if len(subset) > 0:
        print(f"  {PERIOD_LABELS[period]:25s}: mean={subset.mean():.1f}, median={subset.median():.0f}, n={len(subset):,}")
    else:
        print(f"  {PERIOD_LABELS[period]:25s}: no data")